# EgoIndustrial Training Pipeline

This notebook demonstrates the complete training pipeline for EgoIndustrial egocentric action recognition.

## Prerequisites
- Python 3.11+
- CUDA 12.1+ (for GPU training)
- Datasets downloaded: EPIC-KITCHENS-100, Assembly101, HoloAssist
- Weights & Biases account (optional, for logging)

In [ ]:
# Install dependencies
%pip install -e ".[dev]" -q

In [ ]:
# Import libraries
import pytorch_lightning as pl
import torch
from omegaconf import OmegaConf
from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
    RichProgressBar,
)
from pytorch_lightning.loggers import WandbLogger

from egoindustrial.data import build_dataloader, build_datasets
from egoindustrial.data.transforms import get_transforms
from egoindustrial.training.module import EgoIndustrialModule

print(f"PyTorch: {torch.__version__}")
print(f"PyTorch Lightning: {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

We'll use Hydra for configuration management. You can override any config via command line or programmatically.

In [ ]:
# Define training configuration
config = {
    # Global settings
    "seed": 42,
    "num_gpus": 1 if torch.cuda.is_available() else 0,
    "num_workers": 8,
    "precision": "16-mixed",
    "debug": False,

    # Paths
    "paths": {
        "data_root": "/data",  # Change this to your data path
        "output_root": "./outputs",
        "checkpoint_dir": "./outputs/checkpoints",
        "log_dir": "./outputs/logs",
    },

    # Wandb
    "wandb": {
        "project": "egoindustrial",
        "entity": None,  # Your wandb username/team
        "tags": ["notebook"],
        "mode": "offline",  # Change to "online" for wandb sync
    },

    # Dataset
    "dataset": {
        "name": "mixture",  # "epic_kitchens", "assembly101", "holoassist", "mixture"
        "batch_size": 32,
        "num_workers": 8,
        "clip_len": 16,
        "frame_stride": 2,
        "domain_probs": [0.5, 0.3, 0.2],  # For mixture: [epic, assembly, holoassist]
        "drop_last": True,
        "transforms": {
            "clip_len": 16,
            "crop_size": 224,
            "resize_size": 256,
            "is_train": True,
            "model_type": "videomae",
        },
        "datasets": {
            "epic_kitchens": {
                "root": "${paths.data_root}/epic_kitchens",
                "split": "train",
                "clip_len": 16,
                "frame_stride": 2,
                "use_action_labels": True,
            },
            "assembly101": {
                "root": "${paths.data_root}/assembly101",
                "split": "train",
                "clip_len": 16,
                "frame_stride": 2,
            },
            "holoassist": {
                "root": "${paths.data_root}/holoassist",
                "split": "train",
                "clip_len": 16,
                "frame_stride": 2,
            },
        },
    },

    # Model
    "model": {
        "name": "videomae",  # "videomae", "mvitv2", "slowfast", "internvideo2"
        "num_verb_classes": 97,
        "num_noun_classes": 300,
        "num_action_classes": 3806,
        "pretrained": True,
        "dropout": 0.5,
        "freeze_backbone": False,
    },

    # Training
    "train": {
        "max_epochs": 50,
        "gradient_clip": 1.0,
        "accumulate_grad_batches": 1,
    },

    # Optimizer
    "optimizer": {
        "lr": 1e-4,
        "weight_decay": 0.05,
    },

    # Scheduler
    "scheduler": {
        "warmup_epochs": 5,
        "max_epochs": 50,
    },

    # Loss
    "loss": {
        "verb_weight": 1.0,
        "noun_weight": 1.0,
        "action_weight": 1.0,
        "label_smoothing": 0.1,
    },
}

cfg = OmegaConf.create(config)
print(OmegaConf.to_yaml(cfg))

## Data Preparation

Let's build the dataloaders and inspect a batch.

In [ ]:
# Build transforms
transforms = get_transforms(cfg.dataset.transforms)

# Build datasets
train_datasets = build_datasets(cfg.dataset)
val_datasets = build_datasets({**cfg.dataset, "split": "val"})

# Build dataloaders
train_loader = build_dataloader(
    train_datasets,
    batch_size=cfg.dataset.batch_size,
    num_workers=cfg.dataset.num_workers,
    shuffle=True,
    domain_probs=cfg.dataset.get("domain_probs"),
    drop_last=cfg.dataset.get("drop_last", True),
)

val_loader = build_dataloader(
    val_datasets,
    batch_size=cfg.dataset.batch_size,
    num_workers=cfg.dataset.num_workers,
    shuffle=False,
    drop_last=False,
)

print(f"Train datasets: {[len(d) for d in train_datasets]}")
print(f"Val datasets: {[len(d) for d in val_datasets]}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch: {len(val_loader)}")

In [ ]:
# Inspect a batch
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())
print(f"Video shape: {batch['video'].shape}")
print(f"Verb labels: {batch['verb_label'].shape}")
print(f"Noun labels: {batch['noun_label'].shape}")
print(f"Action labels: {batch['action_label'].shape}")
print(f"Domain names: {batch['domain_name'][:5]}")
print(f"Video IDs: {batch['video_id'][:5]}")

## Model Setup

Create the Lightning module with the specified model configuration.

In [ ]:
# Create model
model = EgoIndustrialModule(OmegaConf.to_container(cfg, resolve=True))

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Training Setup

Configure callbacks, logger, and trainer.

In [ ]:
# Callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=cfg.paths.checkpoint_dir,
        filename="epoch={epoch}-val_loss={val/loss_total:.4f}",
        monitor="val/loss_total",
        mode="min",
        save_top_k=3,
        save_last=True,
    ),
    EarlyStopping(monitor="val/loss_total", patience=10, mode="min"),
    LearningRateMonitor(logging_interval="epoch"),
    RichProgressBar(),
]

# Logger
logger = None
if cfg.wandb.mode != "disabled":
    logger = WandbLogger(
        project=cfg.wandb.project,
        entity=cfg.wandb.entity,
        tags=cfg.wandb.tags,
        offline=cfg.wandb.mode == "offline",
    )

In [ ]:
# Trainer
trainer = pl.Trainer(
    max_epochs=cfg.train.max_epochs,
    accelerator="gpu" if cfg.num_gpus > 0 else "cpu",
    devices=cfg.num_gpus,
    precision=cfg.precision,
    callbacks=callbacks,
    logger=logger,
    log_every_n_steps=10,
    check_val_every_n_epoch=1,
    gradient_clip_val=cfg.train.get("gradient_clip", 1.0),
    accumulate_grad_batches=cfg.train.get("accumulate_grad_batches", 1),
    fast_dev_run=cfg.debug,
)

print("Trainer configured successfully!")

## Start Training

Run the training loop. This will take a while depending on your hardware.

In [ ]:
# Train
trainer.fit(model, train_loader, val_loader)

## Resume Training

If training was interrupted, you can resume from the last checkpoint.

In [ ]:
# To resume from last checkpoint:
# trainer.fit(model, train_loader, val_loader, ckpt_path="outputs/checkpoints/last.ckpt")

# Or from a specific checkpoint:
# trainer.fit(model, train_loader, val_loader, ckpt_path="outputs/checkpoints/epoch=10-val_loss=0.1234.ckpt")

## Export Best Model

After training, export the best model for inference.

In [ ]:
# Export to ONNX
from egoindustrial.inference.export_onnx import export_to_onnx, validate_onnx

best_ckpt = "outputs/checkpoints/best.ckpt"  # or "outputs/checkpoints/last.ckpt"
export_to_onnx(
    checkpoint_path=best_ckpt,
    output_path="outputs/model.onnx",
    opset_version=17,
    simplify=True,
)
validate_onnx("outputs/model.onnx", [1, 3, 16, 224, 224])

## TensorRT Export & Benchmark

In [ ]:
# Build TensorRT engine (FP16)
from egoindustrial.inference.tensorrt_engine import build_tensorrt_engine

build_tensorrt_engine(
    onnx_path="outputs/model.onnx",
    engine_path="outputs/model_fp16.engine",
    precision="fp16",
    max_batch_size=32,
    max_workspace_size=1 << 30,
    input_shapes={
        "video": ([1, 3, 16, 224, 224], [16, 3, 16, 224, 224], [32, 3, 16, 224, 224])
    },
)

In [ ]:
# For INT8 (requires calibration data)
from torch.utils.data import DataLoader

from egoindustrial.data import build_datasets
from egoindustrial.data.transforms import get_transforms
from egoindustrial.data.unified_dataloader import ConcatDatasetWithDomain
from egoindustrial.inference.tensorrt_engine import build_tensorrt_engine

# Build calibration dataloader
transform_cfg = cfg.dataset.transforms
transform_cfg["is_train"] = False
transforms = get_transforms(transform_cfg)

val_datasets = build_datasets({**cfg.dataset, "split": "val"})
concat_dataset = ConcatDatasetWithDomain(val_datasets)
calib_loader = DataLoader(
    concat_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)


build_tensorrt_engine(
    onnx_path="outputs/model.onnx",
    engine_path="outputs/model_int8.engine",
    precision="int8",
    max_batch_size=32,
    max_workspace_size=1 << 30,
    calibration_dataloader=calib_loader,
    calibration_cache="outputs/calibration.cache",
)

## Benchmark Results

Run comprehensive benchmark across batch sizes.

In [ ]:
import json

# Benchmark all backends
from egoindustrial.inference.benchmark import run_full_benchmark

# Load PyTorch model for comparison
pytorch_model = EgoIndustrialModule.load_from_checkpoint(best_ckpt)

results = run_full_benchmark(
    pytorch_model=pytorch_model,
    onnx_path="outputs/model.onnx",
    engine_path="outputs/model_int8.engine",
    input_shapes=[(1, 3, 16, 224, 224)],
    batch_sizes=[1, 4, 8, 16, 32],
    warmup=10,
    runs=100,
    output="outputs/benchmark_results.json",
)


with open("outputs/benchmark_results.json") as f:
    results = json.load(f)

for bs_key, res in results.items():
    print(f"\n{bs_key}:")
    for backend, metrics in res.items():
        if isinstance(metrics, dict) and 'mean_ms' in metrics:
            print(f"  {backend}: {metrics['mean_ms']:.2f}ms, {metrics['fps']:.1f} FPS")

## Next Steps

1. **Monitor training** on W&B dashboard
2. **Export best model** to ONNX + TensorRT
3. **Run inference** with `scripts/infer_video.py` or FastAPI server
4. **Deploy** to GCP Cloud Run or Modal

## Useful Commands

```bash
# Resume training
python -m egoindustrial.training.train ckpt_path=outputs/checkpoints/last.ckpt

# Export ONNX
python -m egoindustrial.inference.export_onnx checkpoint=outputs/checkpoints/best.ckpt

# Build TensorRT engine
python -m egoindustrial.inference.tensorrt_engine onnx=outputs/model.onnx

# Run inference server
python -m egoindustrial.inference.server engine=outputs/model_int8.engine
```